# Interactive 4D-Var Data Assimilation with Caravan-MultiMet

This streamlined interactive tutorial demonstrates **gradient-based 4D-Var Data Assimilation (DA)** using the pretrained Google OpenHydroNets model and Caravan-MultiMet weather forecasts (ECMWF HRES).

The notebook is organized into **three focused diagnostic tiers**:

1. **Tier 1: Single-Initialization Event Inspection:** Interactive two-panel diagnostic to inspect the effect of DA hyperparameters ($w$, $\text{LR}$, $\lambda_{\text{bg}}$, target) on a specific storm event and forecast horizon ($t_0$).
2. **Tier 2: 365-Day Continuous Rolling Forecast:** Evaluates operational generalization across all 365 daily forecast rollouts of a calendar year for a fixed parameter set, generating multi-leadtime continuous hydrographs and a performance scorecard ($\Delta\text{NSE}, \Delta\text{KGE}$).
3. **Tier 3: Precipitation Forcing 4D-Var Optimization:** Direct adjustment of meteorological precipitation inputs with decoupled rainfall correction ($W_P$) and streamflow loss ($W_Q$) windows.


In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import datetime
import yaml
import importlib

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xarray as xr

# Add openhydronets repo and tutorial folders to Python path
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2] if (NOTEBOOK_DIR.name == '02_data_assimilation') else NOTEBOOK_DIR.parents[1]
if not (REPO_ROOT / 'googlehydrology').exists():
    REPO_ROOT = Path('/usr/local/google/home/kruparell/da-paper')

for p in [REPO_ROOT, NOTEBOOK_DIR, REPO_ROOT / 'tutorial' / 'notebooks' / 'helpers']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Reload core dependencies in topological order so existing Jupyter kernels
# pick up `ensure_y_hat` in `googlehydrology.modelzoo.head` without restarting
import googlehydrology.utils.cmal_deterministic as cmal_det_mod
import googlehydrology.modelzoo.head as head_mod
import googlehydrology.utils.assimilationconfig as ac_mod
import googlehydrology.evaluation.assimilation as assim_mod
import da_interactive_helpers

importlib.reload(cmal_det_mod)
importlib.reload(head_mod)
importlib.reload(ac_mod)
importlib.reload(assim_mod)
importlib.reload(da_interactive_helpers)

from googlehydrology.utils.config import Config
from googlehydrology.modelzoo.mean_embedding_forecast_lstm import MeanEmbeddingForecastLSTM
from da_interactive_helpers import (
    calc_hydro_metrics,
    fetch_multimet_batch,
    plot_paper_da_hydrograph,
    plot_single_init_da_hydrographs,
    plot_yearly_rolling_hydrographs,
    plot_interactive_precip_da_hydrographs,
    resolve_basin_streamflow_and_attrs,
    run_precip_epoch_sweep,
    run_single_init_da,
    run_yearly_rolling_evaluation,
)

# -----------------------------------------------------------------------------
# Paths & Pretrained Model Initialization
# -----------------------------------------------------------------------------
CARAVAN_DIR = Path('/usr/local/google/home/kruparell/Caravans_V2')
MULTIMET_DIR = Path('/usr/local/google/home/kruparell/Caravans_MultiMet')
PRETRAINED_DIR = REPO_ROOT / 'pretrained-models' / 'google-floodhub-settings-55-epochs-nse-filtered-0.5-85-epochs'
if not PRETRAINED_DIR.exists():
    PRETRAINED_DIR = Path('/usr/local/google/home/kruparell/da-paper/models/google-floodhub-settings-55-epochs-nse-filtered-0.5-85-epochs')

device = torch.device('cpu')

with open(PRETRAINED_DIR / 'config.yml', 'r') as f:
    model_cfg = Config(yaml.safe_load(f))

model = MeanEmbeddingForecastLSTM(model_cfg)
raw_ckpt = torch.load(PRETRAINED_DIR / 'model_epoch085.pt', map_location='cpu', weights_only=False)
clean_ckpt = {k.replace('_orig_mod.', ''): v for k, v in raw_ckpt.items()}
model.load_state_dict(clean_ckpt, strict=True)
model.to(device)
model.eval()

scaler = xr.open_dataset(PRETRAINED_DIR / 'scaler.nc').load()
q_mean = float(scaler['streamflow'].sel(parameter='mean').values)
q_std = float(scaler['streamflow'].sel(parameter='std').values)

PRIMARY_BASIN = 'camels_12451000'
SECONDARY_BASIN = 'GRDC_4118110'

print(f"Pretrained Model Loaded: {type(model).__name__} ({sum(p.numel() for p in model.parameters()):,} parameters)")
print(f"Streamflow Normalization Statistics: q_mean = {q_mean:.4f}, q_std = {q_std:.4f}")
print(f"Data Directories Ready:\n  Caravan:  {CARAVAN_DIR}\n  MultiMet: {MULTIMET_DIR}")

I0000 00:00:1791109377.399193   42964 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[2026-10-04 10:23:00] [SCALER LOAD START] Loading scaler cache from /usr/local/google/home/kruparell/da-paper/models/google-floodhub-settings-55-epochs-nse-filtered-0.5-85-epochs
[2026-10-04 10:23:01] [SCALER LOAD COMPLETE] Loaded scaler cache in 1.37s (Format: Local NetCDF)
Pretrained Model Loaded: MeanEmbeddingForecastLSTM (3,402,832 parameters)
Streamflow Normalization Statistics: q_mean = 1.7772, q_std = 3.3810
Data Directories Ready:
  Caravan:  /usr/local/google/home/kruparell/Caravans_V2
  MultiMet: /usr/local/google/home/kruparell/Caravans_MultiMet


## Tier 1: Single-Initialization 4D-Var Hydrograph Inspection (Two-Panel Diagnostic)

Inspect the immediate impact of 4D-Var Data Assimilation on an individual storm event:
- **Top Panel (Full 365-Day Evaluation Period):** Continuous hydrographs comparing **Observed** ($Q_{\mathrm{obs}}$, black dots), **Pretrained Base / No DA** (dashed blue), and **DA Assimilated** (solid red). Shaded span and dashed lines indicate the active assimilation window ($[t_0 - w, t_0]$), with full-sequence $[\text{NSE}, \text{KGE}]$ reported in the legend.
- **Bottom Panel (Zoomed Event & Forecast Horizon):** Focused view of the assimilation window and 7-day forecast rollout ($t_0 + 1$ to $t_0 + 7$), with a vertical dashed marker at initialization date $t_0$.


In [9]:
# ==============================================================================
# Interactive Widget: Tier 1 Single-Initialization Hydrograph Inspection
# Supports ANY gauge on the fly via Combobox (type any Caravan/MultiMet basin ID)
# ==============================================================================
out_t1 = widgets.Output()

PRESET_BASINS = [
    'camelsbr_40060001',
    'camelscl_8317001',
    'camelscl_7339001',
    'camelscl_5101001',
    'lamah_18481006',
    'hysets_01591700',
    'hysets_02ZJ001',
    'hysets_12101500',
    'camels_12451000',
    'camelsaus_145018A',
    'hysets_02143000',
    'camels_03281500',
    'hysets_02FE017',
    'GRDC_4118110',
    'hysets_05569500',
    'camels_09047700',
]

basin_t1 = widgets.Combobox(
    options=PRESET_BASINS,
    value='camelscl_7339001',
    placeholder='Type or select any basin ID (e.g. camelscl_7339001)',
    ensure_option=False,
    description='Basin ID:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

date_t1 = widgets.DatePicker(
    value=datetime.date(2017, 6, 14),
    description='Init Date (t0):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

target_t1 = widgets.Dropdown(
    options=[
        ('Stage 28 Best Safe: Static + Hindcast (embedded_both)', 'embedded_both'),
        ('Triple (Static + Hindcast + Forecast)', 'embedded_all'),
        ('Dual-Dynamic (Hindcast + Forecast)', 'embedded_both_dynamic'),
        ('Hindcast Embedding (Dynamic)', 'hindcast_embedding'),
        ('Static Catchment Embedding', 'static_embedding'),
        ('Forecast Embedding (Dynamic)', 'forecast_embedding'),
        ('Both Cell States (c_0_hindcast + c_0_forecast)', 'c_both'),
        ('Recurrent Cell State (c_0_hindcast)', 'c_0_hindcast'),
        ('Initial Recurrent State (c_0_forecast)', 'c_0_forecast'),
    ],
    value='embedded_both',
    description='DA Target:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

loss_t1 = widgets.Dropdown(
    options=[
        ('MSE (deterministic - Stage 28 standard)', 'MSE'),
        ('CMAL (probabilistic NLL)', 'CMAL'),
    ],
    value='MSE',
    description='DA Loss:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

win_t1 = widgets.IntSlider(
    value=365,
    min=1,
    max=365,
    step=1,
    description='Window (w):',
    continuous_update=False,
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

lr_t1 = widgets.FloatLogSlider(
    value=0.1,
    base=10,
    min=-6,
    max=1,
    step=0.1,
    description='Learning Rate (LR):',
    continuous_update=False,
    readout_format='.2e',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

epoch_t1 = widgets.IntSlider(
    value=50,
    min=0,
    max=500,
    step=10,
    description='Max Epochs:',
    continuous_update=False,
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

tol_t1 = widgets.Dropdown(
    options=[
        ('None (run full epochs - Recommended)', 0.0),
        ('5% (0.05 relative error)', 0.05),
        ('2% (0.02 relative error)', 0.02),
        ('1% (0.01 relative error)', 0.01),
        ('10% (0.10 relative error)', 0.10),
    ],
    value=0.0,
    description='Tolerance (t0):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

# Added 0.0 (Zero Regularization) for unconstrained in-window overfitting
bg_t1 = widgets.Dropdown(
    options=[
        ('0.0 (Zero Regularization - Overfitting mode)', 0.0),
        ('1.0e-06 (Ultra-light)', 1e-6),
        ('1.0e-04 (Light)', 1e-4),
        ('1.0e-03', 1e-3),
        ('1.0e-02', 1e-2),
        ('1.0e-01', 0.1),
        ('5.0e-01', 0.5),
        ('1.0e+00', 1.0),
        ('5.0e+00', 5.0),
        ('1.0e+01 (Strict)', 10.0),
    ],
    value=0.0,
    description='Reg Weight (bg):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

btn_t1 = widgets.Button(
    description='Run DA & Plot',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='220px', height='38px', font_weight='bold'),
)

# Auto-suggest valid active dates for basins whose Caravan records end in 2018
_BASIN_RECOMMENDED_DATES = {
    'camelsbr_40060001': datetime.date(2017, 3, 15),
    'camelscl_8317001': datetime.date(2017, 8, 15),
    'camelscl_7339001': datetime.date(2017, 6, 14),
    'camelscl_5101001': datetime.date(2017, 2, 1),
    'lamah_18481006': datetime.date(2017, 6, 13),
    'hysets_02ZJ001': datetime.date(2017, 5, 10),
    'hysets_01591700': datetime.date(2017, 5, 10),
}

def _on_basin_change_t1(change):
    b_val = str(change.get('new', '')).strip()
    if b_val in _BASIN_RECOMMENDED_DATES:
        date_t1.value = _BASIN_RECOMMENDED_DATES[b_val]

basin_t1.observe(_on_basin_change_t1, names='value')

def _on_target_change_t1(change):
    new_t = change.get('new')
    if new_t == 'static_embedding':
        lr_t1.value = 1e-2
    elif new_t in ('embedded_both', 'embedded_all'):
        lr_t1.value = 0.1

target_t1.observe(_on_target_change_t1, names='value')

controls_t1 = widgets.VBox([
    widgets.HTML('<h4>Single-Initialization 4D-Var Inspection (Type Any Gauge ID)</h4>'
                 '<p style="font-size:12px;color:#555;margin-top:-6px;">'
                 'Type or select <b>any Caravan/MultiMet gauge ID</b> in <code>Basin ID</code>. '
                 'Now supports <b>Reg Weight (bg) = 0.0</b> for pure unconstrained in-window overfitting tests.</p>'),
    widgets.HBox([basin_t1, date_t1]),
    widgets.HBox([target_t1, win_t1]),
    widgets.HBox([lr_t1, epoch_t1]),
    widgets.HBox([bg_t1, tol_t1]),
    widgets.HBox([loss_t1]),
    widgets.HBox([btn_t1]),
])

def on_run_t1(_=None):
    out_t1.clear_output(wait=True)
    with out_t1:
        b_id = str(basin_t1.value).strip()
        dt_val = date_t1.value
        issue_date_str = dt_val.strftime('%Y-%m-%d') if hasattr(dt_val, 'strftime') else str(dt_val)
        target_raw = target_t1.value
        if target_raw == 'embedded_both_dynamic':
            target = ['hindcast_embedding', 'forecast_embedding']
        elif target_raw == 'embedded_both':
            target = ['static_embedding', 'hindcast_embedding']
        else:
            target = target_raw
        w = int(win_t1.value)
        lr = float(lr_t1.value)
        ep = int(epoch_t1.value)
        bg = float(bg_t1.value)
        tol_val = float(tol_t1.value) if float(tol_t1.value) > 0 else None
        loss_fn = str(loss_t1.value)

        print(f"Running DA: Basin={b_id}, t0={issue_date_str}, Target={target}, Loss={loss_fn}, "
              f"Window={w}d, LR={lr:.2e}, MaxEpochs={ep}, Tol={tol_val}, bg={bg:.1e}...")

        batch, b_stream = fetch_multimet_batch(
            basin_id=b_id,
            issue_date=issue_date_str,
            model_cfg=model_cfg,
            scaler=scaler,
            caravan_dir=CARAVAN_DIR,
            multimet_dir=MULTIMET_DIR,
            device=device,
        )

        dates_raw = batch['date'][0]
        dates_pd = pd.to_datetime(dates_raw)
        obs = b_stream['streamflow'].sel(date=dates_raw).values.astype(np.float32)

        w_start = max(0, 358 - w)
        n_valid_win = int(np.sum(~np.isnan(obs[w_start:358])))
        n_valid_lead = int(np.sum(~np.isnan(obs[358:])))
        if n_valid_win == 0:
            s_all = b_stream['streamflow'].to_series().dropna()
            valid_range = f"{s_all.index.min().strftime('%Y-%m-%d')} to {s_all.index.max().strftime('%Y-%m-%d')}" if len(s_all) else "No valid observations"
            print(f"⚠️ WARNING: {b_id} has 0 valid streamflow observations in the {w}-day assimilation window before {issue_date_str}!")
            print(f"   When all window observations are NaN, 4D-Var cannot compute gradients and returns the unassimilated baseline.")
            print(f"   👉 Active observation period for {b_id}: {valid_range}. Pick an Init Date (t0) inside that range!")
        else:
            print(f"Observation coverage: {n_valid_win}/{358 - w_start} valid days in assimilation window | {n_valid_lead}/7 valid days in forecast horizon.")

        q_base, q_da = run_single_init_da(
            model=model,
            batch=batch,
            target=target,
            window=w,
            lr=lr,
            epochs=ep,
            bg=bg,
            q_mean=q_mean,
            q_std=q_std,
            loss=loss_fn,
            tol=tol_val,
        )

        fig, _ = plot_single_init_da_hydrographs(
            dates=dates_pd,
            obs=obs,
            q_base=q_base,
            q_da=q_da,
            basin_id=b_id,
            issue_date_str=issue_date_str,
            target_name=f'{target_raw}, {loss_fn}',
            window=w,
            w_start_idx=w_start,
            t0_idx=357,
            da_end_idx=358,
            lead_time=7,
        )
        display(fig)
        plt.close(fig)

btn_t1.on_click(on_run_t1)
display(widgets.VBox([controls_t1, out_t1]))

## Tier 1b: Paper-Ready Single-Initialization Hydrograph

A publication-oriented single-panel version of the Tier 1 plot. Differences from the diagnostic widget above:

- **Any gauge**: search all ~24.9k Caravan gauges by ID or name (comma-separated terms are OR-ed, optional country / dataset filter), pick from the matches, or type an ID directly. **Suggest t0** sets the init date so the largest observed peak of the chosen year falls at lead 2 (skipping peaks with poor window coverage).
- **Typed inputs** for the assimilation window length and the displayed time-series length (in days). The plot always ends 7 days after the end of the assimilation window, so the display length only controls how much warm-up is visible.
- **Period labels** (warm-up / assimilation / forecast) are drawn *inside* the axes over the shaded spans, not in the legend.
- **Metrics** are computed over the **forecast horizon only** and shown in the title. Select which ones (default: NSE only - no KGE).
- Toggle the baseline line, AR(1) post-processing line, legend, labels, shading, log-scale y-axis, and set custom labels / title / figure size. Optionally save to PNG/PDF.
- **Re-plot** re-renders from the cached DA run so you can iterate on styling without re-running 4D-Var.
- Defaults are the **Stage 52 Global-Best** DA config `S52_T_w90_lrd0.001_r1.0_bg0.5_tol0.05_ep50` (all three embeddings, $w=90$, LR $=10^{-3}$ for both static and dynamic embeddings, $bg=0.5$, tol $=0.05$, 50 epochs) and the Global-Best post-processor `AR1_postprocess_rho0.65`, drawn as an optional AR(1) error-persistence line over the forecast horizon.

> Run the Tier 1 cell first (it defines `PRESET_BASINS` and recommended dates), then run this cell.

In [ ]:
# ==============================================================================
# Interactive Widget: Tier 1b Paper-Ready Single-Initialization Hydrograph
# Any Caravan gauge (search by ID/name), typed window / display length,
# in-plot period labels, forecast-only metrics in title.
#
# Defaults = Stage 52 Global-Best DA config `S52_T_w90_lrd0.001_r1.0_bg0.5_tol0.05_ep50`
# (T = all three embeddings, w=90d, LR_dyn=0.001 with LR_stat/LR_dyn ratio r=1.0 -> single LR 0.001,
#  bg=0.5, tol=0.05, 50 epochs) and Global-Best post-processor `AR1_postprocess_rho0.65`.
# ==============================================================================
from da_interactive_helpers import (
    ar1_postprocess,
    format_basin_option,
    load_basin_catalogue,
    observation_coverage,
    plot_paper_da_hydrograph,
    search_basin_catalogue,
    suggest_init_date,
)

out_pp = widgets.Output()
_PP_CACHE = {}  # caches last DA run so "Re-plot" can restyle without re-running 4D-Var

_W = {'description_width': '140px'}
_L = widgets.Layout(width='380px')
_LS = widgets.Layout(width='185px')

# ---- Gauge selection: any of the ~24.9k Caravan gauges -------------------------
_CATALOGUE = load_basin_catalogue(CARAVAN_DIR)
search_pp = widgets.Text(
    value='', placeholder='ID or name, comma = OR (e.g. camelscl, 09444500, danube) - press Enter',
    description='Search gauges:', continuous_update=False, style=_W, layout=widgets.Layout(width='775px'))
country_pp = widgets.Text(value='', placeholder='name or ISO code, e.g. Chile / CL (optional)',
                          description='Country filter:', continuous_update=False, style=_W, layout=_L)
dataset_pp = widgets.Dropdown(options=['(all)'] + sorted({b.split('_')[0] for b in _CATALOGUE.index}),
                              value='(all)', description='Dataset:', style=_W, layout=_L)
match_pp = widgets.Dropdown(options=[], description='Matches:', style=_W, layout=widgets.Layout(width='775px'))
basin_pp = widgets.Combobox(
    options=globals().get('PRESET_BASINS', []), value='camelscl_7339001',
    placeholder='Type any basin ID', ensure_option=False,
    description='Basin ID:', style=_W, layout=_L)
date_pp = widgets.DatePicker(value=datetime.date(2017, 6, 14), description='Init Date (t0):', style=_W, layout=_L)
year_pp = widgets.Dropdown(options=[2016, 2017, 2018, 2019, 2020, 2021], value=2017,
                           description='Case-study year:', style=_W, layout=_L)
btn_suggest_pp = widgets.Button(description='Suggest t0 (peak in year)', icon='search', layout=widgets.Layout(width='200px'))
info_pp = widgets.HTML('')

# ---- DA run settings (Stage 52 Global-Best defaults) ---------------------------
target_pp = widgets.Dropdown(
    options=[
        ('Triple (embedded_all) - Stage 52 best', 'embedded_all'),
        ('Static + Hindcast (embedded_both)', 'embedded_both'),
        ('Dual-Dynamic (hindcast + forecast)', 'embedded_both_dynamic'),
        ('Hindcast Embedding', 'hindcast_embedding'),
        ('Static Embedding', 'static_embedding'),
        ('Forecast Embedding', 'forecast_embedding'),
        ('Both Cell States (c_both)', 'c_both'),
        ('Cell State c_0_hindcast', 'c_0_hindcast'),
        ('Cell State c_0_forecast', 'c_0_forecast'),
    ],
    value='embedded_all', description='DA Target:', style=_W, layout=_L)
loss_pp = widgets.Dropdown(options=[('MSE', 'MSE'), ('CMAL', 'CMAL')], value='MSE', description='DA Loss:', style=_W, layout=_L)
win_pp = widgets.BoundedIntText(value=90, min=1, max=358, step=1, description='Window (days):', style=_W, layout=_L)
disp_pp = widgets.BoundedIntText(value=120, min=8, max=365, step=1, description='Display length (days):', style=_W, layout=_L)
lr_pp = widgets.FloatText(value=0.001, description='Learning Rate:', style=_W, layout=_L)
epoch_pp = widgets.BoundedIntText(value=50, min=0, max=1000, step=1, description='Max Epochs:', style=_W, layout=_L)
bg_pp = widgets.FloatText(value=0.5, description='Reg Weight (bg):', style=_W, layout=_L)
tol_pp = widgets.FloatText(value=0.05, description='Tolerance (0=off):', style=_W, layout=_L)


def _update_matches_pp(*_):
    hits = search_basin_catalogue(search_pp.value, CARAVAN_DIR, limit=300,
                                  country=(country_pp.value.strip() or None),
                                  dataset=(None if dataset_pp.value == '(all)' else dataset_pp.value))
    opts = [(format_basin_option(b, r), b) for b, r in hits.iterrows()]
    match_pp.options = opts
    match_pp.description = f'Matches ({len(hits)}{"+" if len(hits) >= 300 else ""}/{len(_CATALOGUE):,}):'
    if opts:
        match_pp.value = opts[0][1]


def _on_match_pp(change):
    if change.get('new'):
        basin_pp.value = str(change['new'])


def _on_basin_change_pp(change):
    b_val = str(change.get('new', '')).strip()
    rec = globals().get('_BASIN_RECOMMENDED_DATES', {})
    if b_val in rec:
        date_pp.value = rec[b_val]
    if b_val in _CATALOGUE.index:
        row = _CATALOGUE.loc[b_val]
        extra = []
        for k, fmt in (('aridity', 'aridity={:.2f}'), ('frac_snow', 'snow frac={:.2f}'),
                       ('gauge_lat', 'lat={:.2f}'), ('gauge_lon', 'lon={:.2f}')):
            if k in row and pd.notna(row[k]):
                extra.append(fmt.format(float(row[k])))
        info_pp.value = (f'<span style="font-size:12px;color:#374151;">{format_basin_option(b_val, row)}'
                         + (f' &middot; {", ".join(extra)}' if extra else '') + '</span>')
    else:
        info_pp.value = f'<span style="font-size:12px;color:#b45309;">{b_val}: not found in Caravan attributes.zarr</span>'


def _on_suggest_pp(_=None):
    b_id = str(basin_pp.value).strip()
    try:
        _, b_stream = resolve_basin_streamflow_and_attrs(b_id, CARAVAN_DIR)
    except KeyError:
        info_pp.value = f'<span style="color:#b91c1c;">{b_id}: no streamflow record found.</span>'
        return
    cov = observation_coverage(b_stream, int(year_pp.value))
    t0 = suggest_init_date(b_stream, year=int(year_pp.value), window=int(win_pp.value))
    if t0 is None:
        cov_all = observation_coverage(b_stream)
        rng = (f"{cov_all['first_valid']:%Y-%m-%d} to {cov_all['last_valid']:%Y-%m-%d}"
               if cov_all['first_valid'] is not None else 'none')
        info_pp.value = (f'<span style="color:#b91c1c;">{b_id}: no usable observations in {year_pp.value} '
                         f'({cov["n_valid"]}/{cov["n_total"]} valid days). Observed record: {rng}.</span>')
        return
    date_pp.value = t0
    info_pp.value = (f'<span style="font-size:12px;color:#065f46;">{b_id}: t0 set to {t0} '
                     f'(largest {year_pp.value} peak at lead 2; {cov["n_valid"]}/{cov["n_total"]} valid obs days in {year_pp.value}).</span>')


search_pp.observe(_update_matches_pp, names='value')
country_pp.observe(_update_matches_pp, names='value')
dataset_pp.observe(_update_matches_pp, names='value')
match_pp.observe(_on_match_pp, names='value')
basin_pp.observe(_on_basin_change_pp, names='value')
btn_suggest_pp.on_click(_on_suggest_pp)
_on_basin_change_pp({'new': basin_pp.value})

# ---- Plot content / styling ----------------------------------------------------
show_obs_pp = widgets.Checkbox(value=True, description='Observed', indent=False, layout=_LS)
show_base_pp = widgets.Checkbox(value=True, description='No-DA baseline', indent=False, layout=_LS)
show_da_pp = widgets.Checkbox(value=True, description='DA line', indent=False, layout=_LS)
show_pp_pp = widgets.Checkbox(value=True, description='AR(1) PP line', indent=False, layout=_LS)
rho_pp = widgets.FloatText(value=0.65, description='AR(1) rho:', style=_W, layout=_L)
show_legend_pp = widgets.Checkbox(value=True, description='Legend', indent=False, layout=_LS)
period_labels_pp = widgets.Checkbox(value=True, description='Period labels in plot', indent=False, layout=_LS)
shade_pp = widgets.Checkbox(value=True, description='Shade periods', indent=False, layout=_LS)
bounds_pp = widgets.Checkbox(value=True, description='Boundary lines', indent=False, layout=_LS)
logy_pp = widgets.Checkbox(value=False, description='Log y-axis', indent=False, layout=_LS)

metrics_pp = widgets.SelectMultiple(
    options=['NSE', 'KGE', 'RMSE', 'Pearson-r'], value=('NSE',), rows=4,
    description='Title metrics (forecast only):', style=_W, layout=_L)
decimals_pp = widgets.BoundedIntText(value=2, min=0, max=4, description='Metric decimals:', style=_W, layout=_L)
title_pp = widgets.Text(value='', placeholder='blank = auto "<basin> | Forecast NSE: ..."', description='Custom title:', style=_W, layout=_L)
prefix_pp = widgets.Text(value='', placeholder='blank = basin id', description='Title prefix:', style=_W, layout=_L)
lbl_warm_pp = widgets.Text(value='Warm-up', description='Warm-up label:', style=_W, layout=_L)
lbl_win_pp = widgets.Text(value='Assimilation window', description='Window label:', style=_W, layout=_L)
lbl_fc_pp = widgets.Text(value='Forecast', description='Forecast label:', style=_W, layout=_L)
leg_obs_pp = widgets.Text(value='Observed', description='Legend: obs', style=_W, layout=_L)
leg_base_pp = widgets.Text(value='No assimilation', description='Legend: baseline', style=_W, layout=_L)
leg_da_pp = widgets.Text(value='With assimilation', description='Legend: DA', style=_W, layout=_L)
leg_pp_pp = widgets.Text(value='AR(1) post-processing', description='Legend: PP', style=_W, layout=_L)
legend_loc_pp = widgets.Dropdown(
    options=['best', 'upper left', 'upper center', 'upper right', 'center left', 'center right',
             'lower left', 'lower center', 'lower right'],
    value='best', description='Legend location:', style=_W, layout=_L)
date_fmt_pp = widgets.Dropdown(
    options=[('01 Jun 2017', '%d %b %Y'), ('Jun 01', '%b %d'), ('2017-06-01', '%Y-%m-%d'), ('Jun 2017', '%b %Y')],
    value='%d %b %Y', description='Date format:', style=_W, layout=_L)
fig_w_pp = widgets.FloatText(value=10.0, description='Fig width (in):', style=_W, layout=_L)
fig_h_pp = widgets.FloatText(value=4.0, description='Fig height (in):', style=_W, layout=_L)
font_pp = widgets.FloatText(value=11.0, description='Font size:', style=_W, layout=_L)
dpi_pp = widgets.BoundedIntText(value=150, min=50, max=600, description='DPI:', style=_W, layout=_L)
ylabel_pp = widgets.Text(value=r'Streamflow ($\mathrm{mm\,d^{-1}}$)', description='Y label:', style=_W, layout=_L)
xlabel_pp = widgets.Text(value='', placeholder='blank = none', description='X label:', style=_W, layout=_L)
save_pp = widgets.Text(value='', placeholder='e.g. /tmp/fig_da_camelscl_7339001.pdf (blank = no save)',
                       description='Save to:', style=_W, layout=widgets.Layout(width='775px'))

btn_run_pp = widgets.Button(description='Run DA & Plot', button_style='primary', icon='play',
                            layout=widgets.Layout(width='200px', height='38px'))
btn_replot_pp = widgets.Button(description='Re-plot (cached DA)', button_style='info', icon='refresh',
                               layout=widgets.Layout(width='200px', height='38px'))

controls_pp = widgets.VBox([
    widgets.HTML('<h4>Paper-Ready Single-Initialization Hydrograph</h4>'
                 '<p style="font-size:12px;color:#555;margin-top:-6px;">'
                 'Type the <b>window length</b> and the <b>display length</b> in days. The plot always ends 7 days after '
                 'the end of the assimilation window. Metrics in the title are for the <b>forecast horizon only</b>.</p>'),
    widgets.HTML('<b>Gauge</b> &mdash; search all Caravan gauges by ID or name, pick a match, or type an ID directly'),
    widgets.HBox([search_pp]),
    widgets.HBox([country_pp, dataset_pp]),
    widgets.HBox([match_pp]),
    widgets.HBox([basin_pp, date_pp]),
    widgets.HBox([year_pp, btn_suggest_pp]),
    info_pp,
    widgets.HTML('<b>DA run</b>'),
    widgets.HBox([target_pp, loss_pp]),
    widgets.HBox([win_pp, disp_pp]),
    widgets.HBox([lr_pp, epoch_pp]),
    widgets.HBox([bg_pp, tol_pp]),
    widgets.HTML('<b>Plot content</b>'),
    widgets.HBox([show_obs_pp, show_base_pp, show_da_pp, show_pp_pp, show_legend_pp]),
    widgets.HBox([rho_pp]),
    widgets.HBox([period_labels_pp, shade_pp, bounds_pp, logy_pp]),
    widgets.HBox([metrics_pp, widgets.VBox([decimals_pp, title_pp, prefix_pp])]),
    widgets.HTML('<b>Labels & style</b>'),
    widgets.HBox([lbl_warm_pp, lbl_win_pp]),
    widgets.HBox([lbl_fc_pp, legend_loc_pp]),
    widgets.HBox([leg_obs_pp, leg_base_pp]),
    widgets.HBox([leg_da_pp, leg_pp_pp]),
    widgets.HBox([date_fmt_pp]),
    widgets.HBox([fig_w_pp, fig_h_pp]),
    widgets.HBox([font_pp, dpi_pp]),
    widgets.HBox([ylabel_pp, xlabel_pp]),
    widgets.HBox([save_pp]),
    widgets.HBox([btn_run_pp, btn_replot_pp]),
])


def _render_pp(res):
    """Renders the paper figure from a cached DA result dict."""
    fig, ax, fc_m = plot_paper_da_hydrograph(
        dates=res['dates'], obs=res['obs'], q_base=res['q_base'], q_da=res['q_da'],
        basin_id=res['basin'], window=res['window'], w_start_idx=res['w_start'],
        t0_idx=357, da_end_idx=358, lead_time=7,
        q_pp=ar1_postprocess(res['obs'], res['q_base'], rho=float(rho_pp.value)),
        display_days=int(disp_pp.value),
        show_obs=show_obs_pp.value, show_baseline=show_base_pp.value, show_da=show_da_pp.value,
        show_pp=show_pp_pp.value,
        title=(title_pp.value or None), title_prefix=(prefix_pp.value or None),
        metrics=tuple(metrics_pp.value), metric_decimals=int(decimals_pp.value),
        period_labels=period_labels_pp.value,
        period_label_names=(lbl_warm_pp.value, lbl_win_pp.value, lbl_fc_pp.value),
        shade_periods=shade_pp.value, show_boundaries=bounds_pp.value,
        show_legend=show_legend_pp.value, legend_loc=legend_loc_pp.value,
        legend_labels={'obs': leg_obs_pp.value, 'base': leg_base_pp.value,
                       'da': leg_da_pp.value, 'pp': leg_pp_pp.value},
        figsize=(float(fig_w_pp.value), float(fig_h_pp.value)), dpi=int(dpi_pp.value),
        date_fmt=date_fmt_pp.value, xlabel=(xlabel_pp.value or None), ylabel=ylabel_pp.value,
        log_y=logy_pp.value, fontsize=float(font_pp.value),
    )
    print("Forecast-horizon metrics (lead 1-7):")
    for k, v in fc_m.items():
        print(f"  {k:>4}: " + ", ".join(f"{m}={val:.3f}" for m, val in v.items()))
    save_path = save_pp.value.strip()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight')
        print(f"Saved figure to {save_path}")
    display(fig)
    plt.close(fig)


def on_run_pp(_=None):
    out_pp.clear_output(wait=True)
    with out_pp:
        b_id = str(basin_pp.value).strip()
        dt_val = date_pp.value
        issue_date_str = dt_val.strftime('%Y-%m-%d') if hasattr(dt_val, 'strftime') else str(dt_val)
        target_raw = target_pp.value
        w = int(win_pp.value)
        lr, ep, bg = float(lr_pp.value), int(epoch_pp.value), float(bg_pp.value)
        tol_val = float(tol_pp.value) if float(tol_pp.value) > 0 else None
        loss_fn = str(loss_pp.value)

        print(f"Running DA: Basin={b_id}, t0={issue_date_str}, Target={target_raw}, Loss={loss_fn}, "
              f"Window={w}d, LR={lr:.2e}, MaxEpochs={ep}, Tol={tol_val}, bg={bg:.1e}...")

        batch, b_stream = fetch_multimet_batch(
            basin_id=b_id, issue_date=issue_date_str, model_cfg=model_cfg, scaler=scaler,
            caravan_dir=CARAVAN_DIR, multimet_dir=MULTIMET_DIR, device=device)
        dates_raw = batch['date'][0]
        dates_pd = pd.to_datetime(dates_raw)
        obs = b_stream['streamflow'].sel(date=dates_raw).values.astype(np.float32)
        w_start = max(0, 358 - w)
        n_valid_win = int(np.sum(~np.isnan(obs[w_start:358])))
        n_valid_lead = int(np.sum(~np.isnan(obs[358:])))
        if n_valid_win == 0:
            cov_all = observation_coverage(b_stream)
            rng = (f"{cov_all['first_valid']:%Y-%m-%d} to {cov_all['last_valid']:%Y-%m-%d}"
                   if cov_all['first_valid'] is not None else 'none')
            print(f"⚠️ WARNING: {b_id} has 0 valid observations in the {w}-day window before {issue_date_str}; DA will return the baseline.")
            print(f"   Observed record: {rng}. Use 'Suggest t0' or pick an Init Date inside that range.")
        else:
            print(f"Observation coverage: {n_valid_win}/{358 - w_start} valid days in window | {n_valid_lead}/7 in forecast horizon.")

        q_base, q_da = run_single_init_da(
            model=model, batch=batch, target=target_raw, window=w, lr=lr, epochs=ep, bg=bg,
            q_mean=q_mean, q_std=q_std, loss=loss_fn, tol=tol_val)

        _PP_CACHE.clear()
        _PP_CACHE.update(dict(dates=dates_pd, obs=obs, q_base=q_base, q_da=q_da,
                              basin=b_id, window=w, w_start=w_start, issue_date=issue_date_str))
        _render_pp(_PP_CACHE)


def on_replot_pp(_=None):
    out_pp.clear_output(wait=True)
    with out_pp:
        if not _PP_CACHE:
            print("No cached DA run yet - click 'Run DA & Plot' first.")
            return
        print(f"Re-plotting cached run: Basin={_PP_CACHE['basin']}, t0={_PP_CACHE['issue_date']}, Window={_PP_CACHE['window']}d")
        _render_pp(_PP_CACHE)


btn_run_pp.on_click(on_run_pp)
btn_replot_pp.on_click(on_replot_pp)
display(widgets.VBox([controls_pp, out_pp]))

# hysets_02357150

# PAPER_SHOWCASE_BASINS = [
#     "camelscl_4703002",  # Base NSE: 0.61 -> DA: 0.94 (SS: +0.85, KGE: 0.80 -> 0.94, Basin Best DA: 0.94)
#     "hysets_06071300",   # Base NSE: 0.65 -> DA: 0.92 (SS: +0.78, KGE: 0.78 -> 0.96, Basin Best DA: 0.92)
#     "hysets_02053200",   # Base NSE: 0.51 -> DA: 0.89 (SS: +0.77, KGE: 0.55 -> 0.93, Basin Best DA: 0.89)
#     "hysets_02357150",   # Base NSE: 0.75 -> DA: 0.94 (SS: +0.74, KGE: 0.81 -> 0.95, Basin Best DA: 0.94)
#     "hysets_02047500",   # Base NSE: 0.70 -> DA: 0.91 (SS: +0.71, KGE: 0.77 -> 0.95, Basin Best DA: 0.93)
#     "camelscl_5721001",  # Base NSE: 0.79 -> DA: 0.94 (SS: +0.69, KGE: 0.83 -> 0.96, Basin Best DA: 0.94)
# ]

## Tier 2: 365-Day Continuous Rolling Forecast Evaluation (N-Day Lead Time Hydrographs)

Once you identify promising hyperparameters in Tier 1, test whether they generalize across an entire operational year (365 daily forecast rollouts):
- Loops across all 365 daily initialization dates in the calendar year.
- Computes operational skill metrics ($\text{NSE}$, $\text{KGE}$, $\Delta\text{NSE}$, $\Delta\text{KGE}$) across all 7 forecast lead times.
- Visualizes continuous rolling daily hydrographs comparing Baseline vs. DA for selected lead times (e.g. Leads 1, 3, 7).


In [3]:
# ==============================================================================
# Interactive Widget: Tier 2 365-Day Continuous Rolling Evaluation
# Features the EXACT same DA parameter options, targets, loss, and tolerance as Tier 1
# Includes a 1-click 'Sync from Tier 1' button to copy tested parameters directly
# ==============================================================================
out_t2 = widgets.Output()

basin_t2 = widgets.Combobox(
    options=PRESET_BASINS,
    value='camelscl_7339001',
    placeholder='Type or select any basin ID (e.g. camelscl_7339001)',
    ensure_option=False,
    description='Basin ID:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

year_t2 = widgets.IntSlider(
    value=2017,
    min=2016,
    max=2020,
    step=1,
    description='Eval Year:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

target_t2 = widgets.Dropdown(
    options=[
        ('Stage 28 Best Safe: Static + Hindcast (embedded_both)', 'embedded_both'),
        ('Triple (Static + Hindcast + Forecast)', 'embedded_all'),
        ('Dual-Dynamic (Hindcast + Forecast)', 'embedded_both_dynamic'),
        ('Hindcast Embedding (Dynamic)', 'hindcast_embedding'),
        ('Static Catchment Embedding', 'static_embedding'),
        ('Forecast Embedding (Dynamic)', 'forecast_embedding'),
        ('Both Cell States (c_0_hindcast + c_0_forecast)', 'c_both'),
        ('Recurrent Cell State (c_0_hindcast)', 'c_0_hindcast'),
        ('Initial Recurrent State (c_0_forecast)', 'c_0_forecast'),
    ],
    value='embedded_both',
    description='DA Target:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

loss_t2 = widgets.Dropdown(
    options=[
        ('MSE (deterministic - Stage 28 standard)', 'MSE'),
        ('CMAL (probabilistic NLL)', 'CMAL'),
    ],
    value='MSE',
    description='DA Loss:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

win_t2 = widgets.IntSlider(
    value=365,
    min=1,
    max=365,
    step=1,
    description='Window (w):',
    continuous_update=False,
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

lr_t2 = widgets.FloatLogSlider(
    value=0.1,
    base=10,
    min=-6,
    max=1,
    step=0.1,
    description='Learning Rate (LR):',
    continuous_update=False,
    readout_format='.2e',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

epoch_t2 = widgets.IntSlider(
    value=50,
    min=0,
    max=500,
    step=10,
    description='Max Epochs:',
    continuous_update=False,
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

tol_t2 = widgets.Dropdown(
    options=[
        ('None (run full epochs - Recommended)', 0.0),
        ('5% (0.05 relative error)', 0.05),
        ('2% (0.02 relative error)', 0.02),
        ('1% (0.01 relative error)', 0.01),
        ('10% (0.10 relative error)', 0.10),
    ],
    value=0.0,
    description='Tolerance (t0):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

# Added 0.0 (Zero Regularization) for unconstrained in-window overfitting
bg_t2 = widgets.Dropdown(
    options=[
        ('0.0 (Zero Regularization - Overfitting mode)', 0.0),
        ('1.0e-06 (Ultra-light)', 1e-6),
        ('1.0e-04 (Light)', 1e-4),
        ('1.0e-03', 1e-3),
        ('1.0e-02', 1e-2),
        ('1.0e-01', 0.1),
        ('5.0e-01', 0.5),
        ('1.0e+00', 1.0),
        ('5.0e+00', 5.0),
        ('1.0e+01 (Strict)', 10.0),
    ],
    value=0.0,
    description='Reg Weight (bg):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

leads_t2 = widgets.SelectMultiple(
    options=[1, 2, 3, 4, 5, 6, 7],
    value=(1, 3, 7),
    description='Plot Leads:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px', height='80px'),
)

prog_t2 = widgets.IntProgress(
    value=0,
    min=0,
    max=365,
    description='Progress:',
    bar_style='info',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

btn_sync_t2 = widgets.Button(
    description='Sync Parameters from Tier 1',
    button_style='info',
    icon='refresh',
    tooltip='Copy the exact basin, target, loss, window, LR, epochs, tol, and bg from Tier 1',
    layout=widgets.Layout(width='240px', height='38px'),
)

btn_t2 = widgets.Button(
    description='Run 365-Day Rolling Evaluation',
    button_style='danger',
    icon='line-chart',
    layout=widgets.Layout(width='280px', height='38px', font_weight='bold'),
)

def _on_basin_change_t2(change):
    b_val = str(change.get('new', '')).strip()
    if b_val == 'camelsbr_40060001':
        year_t2.value = 2018
    elif b_val == 'camelscl_8317001':
        year_t2.value = 2017

basin_t2.observe(_on_basin_change_t2, names='value')

def _on_target_change_t2(change):
    new_t = change.get('new')
    if new_t == 'static_embedding':
        lr_t2.value = 1e-2
    elif new_t in ('embedded_both', 'embedded_all'):
        lr_t2.value = 0.1

target_t2.observe(_on_target_change_t2, names='value')

def on_sync_from_t1(_=None):
    """Copies all evaluated parameters from Tier 1 into Tier 2."""
    basin_t2.value = basin_t1.value
    target_t2.value = target_t1.value
    loss_t2.value = loss_t1.value
    win_t2.value = win_t1.value
    lr_t2.value = lr_t1.value
    epoch_t2.value = epoch_t1.value
    tol_t2.value = tol_t1.value
    bg_t2.value = bg_t1.value
    dt_val = date_t1.value
    if hasattr(dt_val, 'year'):
        if 2016 <= dt_val.year <= 2020:
            year_t2.value = dt_val.year
    out_t2.clear_output(wait=True)
    with out_t2:
        print(f"Synced parameters from Tier 1: Basin={basin_t2.value}, Target={target_t2.value}, "
              f"Loss={loss_t2.value}, w={win_t2.value}d, LR={lr_t2.value:.2e}, MaxEpochs={epoch_t2.value}, "
              f"Tol={tol_t2.value}, bg={bg_t2.value:.1e}, Year={year_t2.value}")

btn_sync_t2.on_click(on_sync_from_t1)

controls_t2 = widgets.VBox([
    widgets.HTML('<h4>365-Day Continuous Rolling DA Evaluation</h4>'
                 '<p style="font-size:12px;color:#555;margin-top:-6px;">'
                 'Executes rolling daily DA across all 365 days of <code>Eval Year</code> using the exact same '
                 'DA options as Tier 1. Now supports <b>Reg Weight (bg) = 0.0</b>.</p>'),
    widgets.HBox([basin_t2, year_t2]),
    widgets.HBox([target_t2, win_t2]),
    widgets.HBox([lr_t2, epoch_t2]),
    widgets.HBox([bg_t2, tol_t2]),
    widgets.HBox([loss_t2, leads_t2]),
    widgets.HBox([btn_sync_t2, btn_t2]),
    widgets.HBox([prog_t2]),
])

def on_run_t2(_=None):
    out_t2.clear_output(wait=True)
    with out_t2:
        b_id = str(basin_t2.value).strip()
        yr = int(year_t2.value)
        target = str(target_t2.value)
        w = int(win_t2.value)
        lr = float(lr_t2.value)
        ep = int(epoch_t2.value)
        bg = float(bg_t2.value)
        tol_val = float(tol_t2.value) if float(tol_t2.value) > 0 else None
        loss_fn = str(loss_t2.value)
        selected_leads = list(leads_t2.value)

        print(f"Launching 365-Day Rolling Evaluation: Basin={b_id}, Year={yr}, Target={target}, "
              f"Loss={loss_fn}, Window={w}d, LR={lr:.2e}, Epochs={ep}, Tol={tol_val}, bg={bg:.1e}...")

        _, b_stream_check = resolve_basin_streamflow_and_attrs(b_id, CARAVAN_DIR)
        s_yr = b_stream_check['streamflow'].sel(date=slice(f'{yr}-01-01', f'{yr}-12-31')).to_series()
        n_valid = int(s_yr.notna().sum())
        if n_valid == 0:
            s_all = b_stream_check['streamflow'].to_series().dropna()
            valid_range = f"{s_all.index.min().strftime('%Y-%m-%d')} to {s_all.index.max().strftime('%Y-%m-%d')}" if len(s_all) else "None"
            print(f"⚠️ WARNING: {b_id} has 0 valid observations in year {yr}! Active record: {valid_range}. Please select a year with valid observations (e.g. 2017).")
            return
        else:
            print(f"Annual observation coverage for {yr}: {n_valid}/365 valid days ({n_valid/365*100:.1f}%).")

        res = run_yearly_rolling_evaluation(
            model=model,
            basin_id=b_id,
            year=yr,
            target=target,
            window=w,
            lr=lr,
            epochs=ep,
            bg=bg,
            model_cfg=model_cfg,
            scaler=scaler,
            caravan_dir=CARAVAN_DIR,
            multimet_dir=MULTIMET_DIR,
            q_mean=q_mean,
            q_std=q_std,
            device=device,
            progress_bar=prog_t2,
            loss=loss_fn,
            tol=tol_val,
        )

        print("\n--- 365-Day Rolling Evaluation Scorecard ---")
        display(res['summary_table'])

        fig_rolling, _ = plot_yearly_rolling_hydrographs(
            rolling_results=res,
            basin_id=b_id,
            year=yr,
            leads_to_plot=selected_leads,
        )
        display(fig_rolling)
        plt.close(fig_rolling)

btn_t2.on_click(on_run_t2)
display(widgets.VBox([controls_t2, out_t2]))

## Tier 3: Precipitation Forcing 4D-Var Optimization (Decoupled $W_P$ / $W_Q$)

Perform gradient-based data assimilation directly on dynamic meteorological precipitation forcings ($\mathbf{p}_{\text{opt}}$):
- **Decoupled Windows:** Tunes precipitation over a larger hydrological memory window ($W_P$, e.g. 30–90 days) while constraining streamflow backpropagation error strictly over recent observations ($W_Q$, e.g. 7–14 days).
- **Physical Clamping:** Enforces non-negative rainfall ($\mathbf{p}_{\text{opt}} \ge 0\text{ mm/day}$).
- **Visual Diagnostics:** Compares unnormalized physical rainfall (Original vs. DA Optimal) across multi-product forcings (HRES, ERA5-Land, IMERG, etc.).


In [4]:
# ==============================================================================
# Interactive Widget: Tier 3 Precipitation 4D-Var Input Optimization
# Supports ANY gauge on the fly via Combobox
# ==============================================================================
out_t3 = widgets.Output()

basin_t3 = widgets.Combobox(
    options=PRESET_BASINS,
    value='camelscl_7339001',
    placeholder='Type or select any basin ID (e.g. camelscl_7339001)',
    ensure_option=False,
    description='Basin ID:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

date_t3 = widgets.DatePicker(
    value=datetime.date(2017, 6, 14),
    description='Issue Date (t0):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

wp_t3 = widgets.Dropdown(
    options=[7, 14, 21, 30, 60, 90, 180, 365],
    value=90,
    description='Precip Window (Wp):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

wq_t3 = widgets.Dropdown(
    options=[3, 7, 14, 21, 30, 90, 180, 365],
    value=14,
    description='Loss Window (Wq):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

lr_t3 = widgets.FloatLogSlider(
    value=1.0,
    base=10,
    min=-2,
    max=3,
    step=0.5,
    description='Learning Rate:',
    continuous_update=False,
    readout_format='.1e',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

bg_t3 = widgets.Dropdown(
    options=[
        ('0.0 (Zero Regularization - Overfitting mode)', 0.0),
        ('1.0e-07', 1e-7),
        ('1.0e-06', 1e-6),
        ('1.0e-05', 1e-5),
        ('1.0e-04', 1e-4),
        ('1.0e-03', 1e-3),
        ('1.0e-02', 1e-2),
        ('1.0e-01', 0.1),
        ('1.0e+00', 1.0),
    ],
    value=0.0,
    description='Bg Reg Weight:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

epoch_t3 = widgets.IntSlider(
    value=10,
    min=5,
    max=50,
    step=5,
    description='Max Epochs:',
    continuous_update=False,
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

btn_t3 = widgets.Button(
    description='Run Precipitation DA',
    button_style='success',
    icon='tint',
    layout=widgets.Layout(width='240px', height='38px', font_weight='bold'),
)

def _on_basin_change_t3(change):
    b_val = str(change.get('new', '')).strip()
    if b_val in _BASIN_RECOMMENDED_DATES:
        date_t3.value = _BASIN_RECOMMENDED_DATES[b_val]

basin_t3.observe(_on_basin_change_t3, names='value')

controls_t3 = widgets.VBox([
    widgets.HTML('<h4>Precipitation 4D-Var Optimization (Decoupled $W_P$ / $W_Q$)</h4>'
                 '<p style="font-size:12px;color:#555;margin-top:-6px;">'
                 'Tunes multi-product precipitation forcing inputs over window $W_P$ against streamflow observations in window $W_Q$. Now supports <b>bg = 0.0</b>.</p>'),
    widgets.HBox([basin_t3, date_t3]),
    widgets.HBox([wp_t3, wq_t3]),
    widgets.HBox([lr_t3, bg_t3]),
    widgets.HBox([epoch_t3, btn_t3]),
])

def on_run_t3(_=None):
    out_t3.clear_output(wait=True)
    with out_t3:
        b_id = str(basin_t3.value).strip()
        dt_val = date_t3.value
        issue_date_str = dt_val.strftime('%Y-%m-%d') if hasattr(dt_val, 'strftime') else str(dt_val)
        wp = int(wp_t3.value)
        wq = int(wq_t3.value)
        lr = float(lr_t3.value)
        bg = float(bg_t3.value)
        max_ep = int(epoch_t3.value)

        print(f"Running Precip DA: Basin={b_id}, t0={issue_date_str}, Wp={wp}d, Wq={wq}d, "
              f"LR={lr:.1e}, bg={bg:.1e}, Epochs={max_ep}...")

        batch, b_stream = fetch_multimet_batch(
            basin_id=b_id,
            issue_date=issue_date_str,
            model_cfg=model_cfg,
            scaler=scaler,
            caravan_dir=CARAVAN_DIR,
            multimet_dir=MULTIMET_DIR,
            device=device,
        )

        p_cfg = {
            'learning_rate': lr,
            'precip_window': wp,
            'observation_window': wq,
            'bg_regularization_weight': bg,
        }

        epochs_to_run = sorted(list({0, 5, max_ep}))
        ep_preds, p_evol = run_precip_epoch_sweep(
            model=model,
            batch_data=batch,
            p_cfg=p_cfg,
            epoch_list=epochs_to_run,
            q_mean=q_mean,
            q_std=q_std,
            device=device,
        )

        dates_raw = batch['date'][0]
        obs = b_stream['streamflow'].sel(date=dates_raw).values.astype(np.float32)

        fig, _ = plot_interactive_precip_da_hydrographs(
            dates_full=dates_raw,
            obs_full=obs,
            ep_preds=ep_preds,
            precip_evol=p_evol,
            da_start_idx=max(0, 358 - wp),
            loss_start_idx=max(0, 358 - wq),
            da_end_idx=358,
            basin_id=b_id,
            issue_date=issue_date_str,
            lr=lr,
            bg=bg,
            precip_window=wp,
            loss_window=wq,
            scaler=scaler,
        )
        display(fig)
        plt.close(fig)

btn_t3.on_click(on_run_t3)
display(widgets.VBox([controls_t3, out_t3]))